<a href="https://colab.research.google.com/github/rashdiwsl/HPC-CW/blob/main/Question_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir Q2_MPI

In [2]:
!apt-get install -y openmpi-bin libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [3]:
%%writefile Q2_MPI/mpi_transpose.cpp

#include <mpi.h>
#include <iostream>
using namespace std;

#define N 1024   // Size of the matrix (1024 x 1024)

int main(int argc, char** argv) {

    // MPI environment
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    // Get process number

    MPI_Comm_size(MPI_COMM_WORLD, &size);
    // Get total number of processes

    int rows = N / size;   \
    // Number of rows each process will handle

    // statics uder to prevent overflow and handle large metrics
    static int matrix[N][N];

    // Original matrix
    static int local[N][N];

    // Part of matrix received by each process
    static int transpose[N][N];  // Final transposed matrix

    // Only process 0 fills the original matrix
    if (rank == 0) {
        for(int i = 0; i < N; i++) {
            for(int j = 0; j < N; j++) {
                matrix[i][j] = i + j;
            }
        }
    }

    // Distribute matrix rows equally to all processes
    MPI_Scatter(matrix, rows * N, MPI_INT,
                local, rows * N, MPI_INT,
                0, MPI_COMM_WORLD);

    // Each process transposes its assigned rows
    for(int i = 0; i < rows; i++) {
        for(int j = 0; j < N; j++) {
            transpose[j][rank * rows + i] = local[i][j];
        }
    }

    // Collect transposed parts back to process 0
    MPI_Gather(&transpose[0][rank * rows], rows * N, MPI_INT,
               transpose, rows * N, MPI_INT,
               0, MPI_COMM_WORLD);

    // Process 0 prints a small part of the transposed matrix
    if (rank == 0) {
        cout << "Transpose Snippet (5x5):" << endl;
        for(int i = 0; i < 5; i++) {
            for(int j = 0; j < 5; j++) {
                cout << transpose[i][j] << " ";
            }
            cout << endl;
        }
    }

    // End MPI
    MPI_Finalize();
    return 0;
}


Writing Q2_MPI/mpi_transpose.cpp


In [4]:
!mpic++ Q2_MPI/mpi_transpose.cpp -o Q2_MPI/mpi_transpose

In [5]:
!mpirun --allow-run-as-root --oversubscribe -np 4 Q2_MPI/mpi_transpose

Transpose Snippet (5x5):
0 1 2 3 4 
1 2 3 4 5 
2 3 4 5 6 
3 4 5 6 7 
4 5 6 7 8 
